In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from src.target import add_target, get_resolve_loan_status

df = pd.read_parquet("data/interim/accepted_2007_to_2018Q4.parquet")

df = df.drop(columns=["member_id", "id"])

resolved_status = get_resolve_loan_status(df)

resolved_status_with_target = add_target(resolved_status)

X = resolved_status_with_target.select_dtypes(include="number").drop(columns=["target"])
y = resolved_status_with_target["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = HistGradientBoostingClassifier(random_state=42)

model.fit(X_train, y_train)


proba = model.predict_proba(X_test)[:, 1]

print(f"ROC AUC Score: {roc_auc_score(y_test, proba):.2%}")


/Users/alexanderhoyskel/VSCode/credit-default
['/Library/Frameworks/Python.framework/Versions/3.11/lib/python311.zip', '/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11', '/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload', '', '/Users/alexanderhoyskel/VSCode/credit-default/.venv/lib/python3.11/site-packages']
ROC AUC Score: 99.91%


# Understanding Week 2

## Filtering

The same logic used in week 1 for excluding non-known `loan_status` outcomes. Added a `target` integer binary column for weather a record is defaulted or not.

## Test and Training Data

`X` is the numeric columns from the table excluding `target` column. While `y` only includes the target column `target`. These two are passed to `train_test_split`, which returns training and testing data. The test size is 20%. `train_test_split` is used so we don't test and train our model on the same data.

## Model

We are using `HistGradientBoostingClassifier`, which is a histogram-based gradient boosted decision tree model. The model is fit with the train data. Then it predicts the predicted probability on the test data. To the get AUC (Area Under the ROC Curve) we use `roc_auc_score`. The model scores ~99.91%, which is extremely high. This is because of leakage. Some columns are not known to a lender when the loan is agreed upon which makes the model score higher than it should.


In [2]:
column_names = X_test.columns
for i, col in enumerate(column_names):
    mask = X_test[col].notna()
    prob = roc_auc_score(y_test[mask], X_test.loc[mask, col])
    score = max(prob, 1 - prob)
    if score > 0.6:
        print(f"{i}: {col}")


3: int_rate
20: total_pymnt
21: total_pymnt_inv
22: total_rec_prncp
25: recoveries
26: collection_recovery_fee
27: last_pymnt_amnt
28: last_fico_range_high
29: last_fico_range_low
34: dti_joint
91: sec_app_fico_range_low
92: sec_app_fico_range_high
94: sec_app_mort_acc
101: sec_app_mths_since_last_major_derog
103: hardship_amount
105: hardship_dpd
107: hardship_payoff_balance_amount
109: settlement_amount
110: settlement_percentage
111: settlement_term


## Leakage Investigation

To find the most leaky column, we loop through the columns and check their AUC score. The columns that are check are the numeric columns. Columns with scores above 70% or below 30% are flagged. We can see that they unknown factor to a lender when initiating the loan. This supports the statements in [Project Description](../project-description.md) about what columns to drop.

## Columns to Drop

After going through the full column list, the columns to drop became clear. These columns are not known at origination.

- **Payments & recoveries** (`total_pymnt`, `total_pymnt_inv`, `total_rec_prncp`, `total_rec_int`, `total_rec_late_fee`, `recoveries`, `collection_recovery_fee`) — only exist once payments have actually occurred during the loan's life.
- **Last-payment/timing** (`last_pymnt_d`, `last_pymnt_amnt`, `next_pymnt_d`, `last_credit_pull_d`) — describe events during loan servicing, not at approval.
- **Outstanding balance** (`out_prncp`, `out_prncp_inv`) — reflects loan state after time has passed. Interestingly, it's ~constant zero for both outcomes in the resolved set — only _currently active_ loans (already filtered out in week 1) carry a nonzero balance, so this column is reasoning-leaky but empirically inert here.
- **Updated FICO** (`last_fico_range_high`, `last_fico_range_low`) — a re-pull after origination, not the original underwriting score used to approve the loan.
- **Settlement/hardship** (`hardship_*`, `settlement_*`, `debt_settlement_flag*`, `payment_plan_start_date`, `orig_projected_additional_accrued_interest`) — only populated for loans that entered financial distress after the loan began.


In [3]:
from src.leakage import LEAKAGE_COLUMNS

resolved_with_dropped_columns = resolved_status_with_target.drop(
    columns=LEAKAGE_COLUMNS
)

X_with_dropped_columns = resolved_with_dropped_columns.select_dtypes(
    include="number"
).drop(columns=["target"])


X_train, X_test, y_train, y_test = train_test_split(
    X_with_dropped_columns, y, test_size=0.2, random_state=42, stratify=y
)
model = HistGradientBoostingClassifier(random_state=42)

model.fit(X_train, y_train)


proba_2 = model.predict_proba(X_test)[:, 1]

print(
    f"ROC AUC Score after dropping leakage from columns: {roc_auc_score(y_test, proba_2):.2%}"
)

ROC AUC Score after dropping leakage from columns: 72.56%


## Reflection After Dropping Leakage

After dropping the leakage columns the AUC score falls from 99.91% to ~72.56%, which is a drop of ~27 percentage points.
